In [1]:
# Import necessary libraries
import os
import glob
import json

import numpy as np
from tqdm import tqdm


In [34]:
BASE_FOLDER_PTH = "../../../../data/people_detection"
DATASET_NAME = "filtered_v1_10_sec"
BASE_PTH = os.path.join(BASE_FOLDER_PTH, DATASET_NAME)

SUBSET = "algotest"
IMGS_PTH = os.path.join(BASE_PTH, "images", SUBSET)
LABELS_PTH = os.path.join(BASE_PTH, "labels", SUBSET)

img_paths = np.array(sorted(glob.glob(os.path.join(IMGS_PTH, "*"))))
label_paths = np.array(sorted(glob.glob(os.path.join(LABELS_PTH, "*"))))

log_pth = f"log_{DATASET_NAME}_{SUBSET}.json"

video_names = [os.path.basename(i)[:-10] for i in img_paths]
uvideo_names, inverse_mapping = np.unique(video_names, return_inverse=True)

len(uvideo_names), len(video_names)

(477, 9572)

In [36]:
indexes = {
    "box": 0,
    "size": 1,
    "pos": 2,
}

with open (log_pth, "rt") as f:
    log_data = json.load(f)

is_viewed = log_data["is_viewed"]
is_filtered = log_data["is_filtered"]
current_index = log_data["current_index"]

print(len(is_viewed), current_index)
sum(is_viewed) == len(is_viewed), current_index == len(is_viewed) -1 

477 2


(False, False)

In [37]:
incorrect_boxes = np.array([i[indexes["size"]] for i in is_filtered])
print("size", sum(incorrect_boxes), sum(~incorrect_boxes))

incorrect_boxes = np.array([i[indexes["pos"]] for i in is_filtered])
print("pos", sum(incorrect_boxes), sum(~incorrect_boxes))

incorrect_boxes = np.array([i[indexes["box"]] for i in is_filtered])
print("box", sum(incorrect_boxes), sum(~incorrect_boxes))

incorrect_boxes = np.array([i[indexes["box"]] | i[indexes["pos"]] for i in is_filtered])
print("box | pos", sum(incorrect_boxes), sum(~incorrect_boxes))

size 0 477
pos 0 477
box 0 477
box | pos 0 477


In [38]:
filtered_frames_idxs = np.in1d(video_names, uvideo_names[incorrect_boxes])
filtered_frames_idxs.sum(), len(video_names)

(0, 9572)

In [39]:
np.array(label_paths)[~filtered_frames_idxs].shape

(9572,)

In [40]:
with open(f"filtered_frames_{DATASET_NAME}_{SUBSET}.json", "wt") as f:
    json.dump({
        "incorrect_count": int(filtered_frames_idxs.sum()),
        "correct_count": len(img_paths) - int(filtered_frames_idxs.sum()),
        "imgs": np.array(img_paths)[~filtered_frames_idxs].tolist(),
        "labels": np.array(label_paths)[~filtered_frames_idxs].tolist(),
    }, f, indent=4)